In [77]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

In [78]:
class SentimentFormate(BaseModel):
    sentiment: Literal['positive', 'negative']


class DiagnosisFormate(BaseModel):
    issue_type: Literal['UX', 'Performance', 'Bug', 'Support', 'Other'] = Field(description="The category of the review mentioned in the review")
    tone: Literal['angry', 'frustrated', 'disappointed', 'calm'] = Field(description="The emotional tone express by the user")
    urgency: Literal['low', 'medium', 'high'] = Field(description="How urgent or critical the issue appears to be")



llm = ChatOllama(
    model='llama3.1',
    max_tokens=150,
)

sentiment_structured_llm = llm.with_structured_output(SentimentFormate)
diagnosis_structured_llm = llm.with_structured_output(DiagnosisFormate)





class ReviewState(TypedDict):       # for graph
    review_text: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict
    response: str
    


In [79]:
def find_sentiment(state: ReviewState):
    prompt = f"Extract the sentiment of the following review of user. \n{state['review_text']}"
    sentiment =  sentiment_structured_llm.invoke(prompt).sentiment

    return {'sentiment': sentiment}



def positive_response(state: ReviewState):
    prompt = f"Write a thank you message to following user's positive review in 2 to 3 lines. \n{state['review_text']} \nAlso ask user to give feedback on our website."
    response = llm.invoke(prompt).content

    return {'response': response}



def run_diagnosis(state: ReviewState):
    prompt = f"Diagnose the following negative review. \n{state['review_text']} \nReturn issue_type, tone and urgency"
    response  = diagnosis_structured_llm.invoke(prompt)

    return {'diagnosis': response.model_dump()}     # .model_dump() is a Pydantic method used to convert a Pydantic model into a normal Python dict.



def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
User had a {diagnosis['issue_type']} issue, sounded {diagnosis['tone']}, and marked urgency as {diagnosis['urgency']}
review: {state['review_text']}
Write an empathetic, helpful resolution message in 2 to 3 lines."""

    response = llm.invoke(prompt).content


    return {'response': response}




def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return "positive_response"
    else:  # negative
        return "run_diagnosis"

In [80]:
graph = StateGraph(ReviewState)


graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)


graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)


workflow = graph.compile()

In [ ]:
# positive review
# initial_state = {'review_text': "I’m really happy with my experience! The product quality was excellent, delivery was fast, and everything arrived exactly as described. The customer service team was also very helpful and friendly. I would definitely recommend this product to others."}


# negative review
initial_state = {'review_text': "I’m extremely frustrated with this application. I’ve been trying to log in using credentials, but it keeps rejecting my correct username and password. I’ve double-checked my credentials multiple times, and they work elsewhere, but the application still returns an authentication error. This is preventing me from accessing the system and needs to be fixed as soon as possible."}

final_state = workflow.invoke(initial_state)

In [88]:
print(final_state)

{'review_text': 'I’m extremely frustrated with this application. I’ve been trying to log in using basic authentication, but it keeps rejecting my correct username and password. I’ve double-checked my credentials multiple times, and they work elsewhere, but the application still returns an authentication error. This is preventing me from accessing the system and needs to be fixed as soon as possible.', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'}, 'response': '"I understand how frustrating this must be for you! I\'d like to escalate your issue to our development team to investigate the authentication error and resolve it immediately. In the meantime, I can offer temporary access via a different login method if that would help."'}
